# EDA — Icentia11k Continuous ECG

## Amostra reprodutível por paciente

O Icentia11k contém cerca de 11.000 pacientes e está organizado em pastas de paciente, por exemplo `p09/p09000/`. O ficheiro `RECORDS` na raiz lista as pastas de pacientes; o `RECORDS` de cada paciente lista os seus segmentos. Cada segmento WFDB tem ficheiros `.dat` (sinal), `.hea` (header) e `.atr` (anotações). A página oficial descreve segmentos com 1.048.577 amostras a 250 Hz e até cerca de 50 segmentos por paciente.

Este notebook seleciona **20 pacientes** (cerca de 0,18% dos 11.000) de forma aleatória e reproduzível, e descarrega todos os segmentos apenas desses pacientes. A estimativa inicial é de cerca de 2 GiB, mas o volume exato varia com o número de segmentos por paciente. A seleção/estimativa não descarrega os sinais; a transferência só começa depois de ativar explicitamente a confirmação na célula seguinte. O download é retomável: ficheiros já completos são ignorados e ficheiros `.part` podem continuar por Range request.

A amostra é ao nível do paciente; não corresponde a 10% dos batimentos nem a uma análise do dataset inteiro. As conclusões do EDA devem ser descritas como referentes aos 20 pacientes selecionados. Consultar e cumprir a licença e as condições de utilização indicadas pelo PhysioNet.

In [ ]:
from pathlib import Path
import re
import shutil

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

BASE_URL = "https://physionet.org/files/icentia11k-continuous-ecg/1.0/"
N_PATIENTS = 20
RANDOM_SEED = 2026
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "DataPreparation").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "DataPreparation").is_dir():
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (pasta DataPreparation).")
DOWNLOAD_DIR = PROJECT_ROOT / "datasets" / "icentia11k_sample_20"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_CSV = DOWNLOAD_DIR / "selected_patients.csv"

session = requests.Session()
response = session.get(BASE_URL + "RECORDS", timeout=60)
response.raise_for_status()
patient_dirs = sorted({
    line.strip().strip("/")
    for line in response.text.splitlines()
    if re.fullmatch(r"p\d{2}/p\d{5}/?", line.strip())
})
if len(patient_dirs) < N_PATIENTS:
    raise ValueError(f"O índice oficial tem apenas {len(patient_dirs)} pacientes.")

# Reutilizar a seleção persistida para que reexecutar o notebook não altere a amostra.
if SAMPLE_CSV.exists():
    selected = pd.read_csv(SAMPLE_CSV)["patient_dir"].astype(str).tolist()
    if len(selected) != N_PATIENTS or len(set(selected)) != N_PATIENTS:
        raise ValueError(f"{SAMPLE_CSV} não contém exatamente {N_PATIENTS} pacientes únicos.")
else:
    rng = np.random.default_rng(RANDOM_SEED)
    selected = sorted(rng.choice(patient_dirs, size=N_PATIENTS, replace=False).tolist())
    pd.DataFrame({"patient_dir": selected}).to_csv(SAMPLE_CSV, index=False)

if not set(selected).issubset(patient_dirs):
    raise ValueError("A seleção persistida contém pastas que não aparecem no índice do PhysioNet.")

print(f"Pacientes disponíveis no índice: {len(patient_dirs):,}")
print(f"Pacientes selecionados: {len(selected):,}")
print(f"Semente: {RANDOM_SEED}")
print(f"Lista persistida em: {SAMPLE_CSV}")
print("Exemplos:", selected[:10])

## Estimar o volume antes de descarregar

A célula seguinte consulta apenas os pequenos ficheiros `RECORDS` de cada paciente selecionado para contar segmentos. Não descarrega sinais. Cada `.dat` observado no índice oficial ocupa cerca de 2 MiB; os headers e anotações também ocupam espaço. A estimativa é aproximada e a célula de download exige uma confirmação explícita.

In [ ]:
patient_records = {}
for patient_dir in tqdm(selected, desc="A obter listas de segmentos"):
    records_path = DOWNLOAD_DIR / patient_dir / "RECORDS"
    if records_path.is_file():
        records = [line.strip() for line in records_path.read_text().splitlines() if line.strip()]
    else:
        url = f"{BASE_URL}{patient_dir}/RECORDS"
        r = session.get(url, timeout=60)
        r.raise_for_status()
        records = [line.strip() for line in r.text.splitlines() if line.strip()]
        records_path.parent.mkdir(parents=True, exist_ok=True)
        records_path.write_text("\n".join(records) + "\n")
    prefix = patient_dir.split("/")[-1]
    if any(not re.fullmatch(re.escape(prefix) + r"_s\d+", name) for name in records):
        raise ValueError(f"Formato inesperado na lista de {patient_dir}")
    patient_records[patient_dir] = records

n_segments = sum(len(records) for records in patient_records.values())
estimated_dat_gib = n_segments * 2_097_154 / (1024 ** 3)
estimated_with_annotations_gib = n_segments * 2_200_000 / (1024 ** 3)
free_gib = shutil.disk_usage(DOWNLOAD_DIR).free / (1024 ** 3)
print(f"Segmentos nos {N_PATIENTS:,} pacientes: {n_segments:,}")
print(f"Estimativa só dos .dat: {estimated_dat_gib:.1f} GiB")
print(f"Estimativa com headers/anotações (aproximada): {estimated_with_annotations_gib:.1f} GiB")
print(f"Espaço livre na unidade: {free_gib:.1f} GiB")
display(pd.Series({p: len(v) for p, v in patient_records.items()}, name="segmentos_por_paciente").describe().to_frame())

if free_gib < estimated_with_annotations_gib * 1.1:
    print("AVISO: espaço livre insuficiente para a estimativa mais 10% de margem; não iniciar o download.")

## Download seletivo e retomável

Por segurança, `CONFIRM_DOWNLOAD` começa como `False`. Só alterá-lo para `True` depois de verificar a estimativa e o espaço livre. O download guarda os segmentos dentro das pastas originais `pXX/pXXXXX/` e obtém os três ficheiros WFDB de cada segmento. Se a transferência for interrompida, executar novamente: ficheiros completos são reutilizados e os temporários tentam continuar.

In [ ]:
CONFIRM_DOWNLOAD = False  # Alterar para True apenas após rever a estimativa de armazenamento.

if not CONFIRM_DOWNLOAD:
    print("Download não iniciado. Confirma o espaço necessário e define CONFIRM_DOWNLOAD = True.")
else:
    def download_file(url, destination):
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists() and destination.stat().st_size > 0:
            return "already_downloaded"

        partial = destination.with_name(destination.name + ".part")
        for attempt in range(6):
            offset = partial.stat().st_size if partial.exists() else 0
            headers = {"Range": f"bytes={offset}-"} if offset else {}
            expected_size = None
            try:
                with get_thread_session().get(url, headers=headers, stream=True, timeout=(30, 180)) as r:
                    if r.status_code in {429, 500, 502, 503, 504}:
                        r.close()
                        if attempt == 5:
                            r.raise_for_status()
                        time.sleep(min(2 ** attempt, 30))
                        continue
                    if offset and r.status_code == 206:
                        content_range = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+|\*)", r.headers.get("Content-Range", ""))
                        if not content_range or int(content_range.group(1)) != offset:
                            raise IOError(f"Content-Range inesperado para {url}")
                        mode = "ab"
                        if content_range.group(3) != "*":
                            expected_size = int(content_range.group(3))
                    else:
                        mode = "wb"
                        if r.headers.get("Content-Length") is not None:
                            expected_size = int(r.headers["Content-Length"])
                    r.raise_for_status()
                    with partial.open(mode) as f:
                        for chunk in r.iter_content(chunk_size=1024 * 1024):
                            if chunk:
                                f.write(chunk)

                actual_size = partial.stat().st_size if partial.exists() else 0
                if actual_size == 0 or (expected_size is not None and actual_size != expected_size):
                    raise requests.exceptions.ChunkedEncodingError(
                        f"Transferência incompleta: {actual_size} de {expected_size} bytes ({url})"
                    )
                partial.replace(destination)
                return "downloaded"
            except (
                requests.exceptions.ChunkedEncodingError,
                requests.exceptions.ConnectionError,
                requests.exceptions.Timeout,
            ):
                if attempt == 5:
                    raise
                time.sleep(min(2 ** attempt, 30))
        raise RuntimeError(f"Não foi possível concluir {url}")

    from concurrent.futures import ThreadPoolExecutor, as_completed
    from threading import local
    import time

    thread_state = local()

    def get_thread_session():
        if not hasattr(thread_state, "session"):
            thread_state.session = requests.Session()
        return thread_state.session

    download_log = []
    files_to_download = [
        (patient_dir, record, extension)
        for patient_dir, records in patient_records.items()
        for record in records
        for extension in ("hea", "dat", "atr")
    ]
    def download_item(item):
        patient_dir, record, extension = item
        relative_path = f"{patient_dir}/{record}.{extension}"
        result = download_file(BASE_URL + relative_path, DOWNLOAD_DIR / relative_path)
        return {"path": relative_path, "status": result}

    progress_path = DOWNLOAD_DIR / "download_progress.csv"
    for start in range(0, len(files_to_download), 128):
        batch = files_to_download[start:start + 128]
        with ThreadPoolExecutor(max_workers=4) as executor:
            futures = [executor.submit(download_item, item) for item in batch]
            for future in tqdm(as_completed(futures), total=len(futures), desc="A descarregar ficheiros"):
                download_log.append(future.result())
        pd.DataFrame(download_log).to_csv(progress_path, index=False)
        print(f"Progresso: {min(start + len(batch), len(files_to_download)):,}/{len(files_to_download):,} ficheiros verificados")
        download_log.clear()

    missing = [
        f"{patient_dir}/{record}.{extension}"
        for patient_dir, records in patient_records.items()
        for record in records
        for extension in ("hea", "dat", "atr")
        if not (DOWNLOAD_DIR / patient_dir / f"{record}.{extension}").is_file()
    ]
    if missing:
        raise IOError(f"Download incompleto: {len(missing)} ficheiros em falta. Exemplo: {missing[:5]}")
    print(f"Download concluído e validado: {len(files_to_download):,} ficheiros dos {N_PATIENTS} pacientes.")

## EDA da amostra local

Instalar `wfdb` no ambiente do notebook se ainda não estiver disponível (`%pip install wfdb`). Depois de concluir o download, a célula seguinte percorre os segmentos selecionados um de cada vez, sem carregar todos os sinais simultaneamente em memória. Resume dimensões, duração, estatísticas de amplitude e rótulos das anotações de batimentos; as anotações de ritmo são guardadas como valores observados para interpretação posterior.

In [ ]:
from collections import Counter

try:
    import wfdb
except ImportError as exc:
    raise ImportError("Instala wfdb no kernel com %pip install wfdb e volta a executar esta célula.") from exc

feature_rows = []
beat_counts = Counter()
rhythm_region_counts = Counter()
rhythm_duration_samples = Counter()
local_records = []
for patient_dir, records in patient_records.items():
    for record in records:
        record_path = DOWNLOAD_DIR / patient_dir / record
        missing_files = [ext for ext in ("hea", "dat", "atr") if not Path(str(record_path) + f".{ext}").is_file()]
        if missing_files:
            raise FileNotFoundError(f"Ficheiros em falta para {record}: {missing_files}")
        rec = wfdb.rdrecord(str(record_path))
        ann = wfdb.rdann(str(record_path), "atr")
        signal = rec.p_signal
        if signal is None:
            raise ValueError(f"WFDB não devolveu sinal físico para {record_path}")
        signal = np.asarray(signal, dtype=np.float64)
        if not np.isfinite(signal).all():
            raise ValueError(f"Valores não finitos encontrados em {record_path}")

        symbols = Counter(symbol for symbol in ann.symbol if str(symbol).strip() not in {"+", "~"})
        beat_counts.update(symbols)
        rhythm_regions = []
        active_rhythm = None
        rhythm_start = None
        for sample, raw_note in zip(ann.sample, ann.aux_note):
            note = str(raw_note).strip()
            if not note or note == "None":
                continue
            if note.startswith("("):
                if active_rhythm is not None and rhythm_start is not None:
                    rhythm_duration_samples[active_rhythm] += int(sample) - rhythm_start
                active_rhythm = note[1:].strip().rstrip(")").strip()
                rhythm_start = int(sample)
                rhythm_regions.append({"sample": int(sample), "rhythm": active_rhythm})
                rhythm_region_counts[active_rhythm] += 1
            elif note == ")" and active_rhythm is not None and rhythm_start is not None:
                rhythm_duration_samples[active_rhythm] += int(sample) - rhythm_start
                active_rhythm = None
                rhythm_start = None
        if active_rhythm is not None and rhythm_start is not None:
            rhythm_duration_samples[active_rhythm] += signal.shape[0] - rhythm_start
        n_samples, n_leads = signal.shape
        fs = float(rec.fs)
        feature_rows.append({
            "patient": patient_dir.split("/")[-1],
            "record": record,
            "n_samples": n_samples,
            "n_leads": n_leads,
            "fs_hz": fs,
            "duration_hours": n_samples / fs / 3600,
            "mean": float(signal.mean()),
            "std": float(signal.std()),
            "min": float(signal.min()),
            "max": float(signal.max()),
            "peak_to_peak": float(np.ptp(signal)),
            "n_annotations": len(ann.symbol),
            "n_rhythm_regions": len(rhythm_regions),
        })
        local_records.append(record)
        del signal, rec, ann

features = pd.DataFrame(feature_rows)
if features.empty:
    raise FileNotFoundError("Não encontrei segmentos completos. Executa primeiro o download confirmado.")
features.to_csv(DOWNLOAD_DIR / "eda_segment_features.csv", index=False)
print(f"Pacientes com segmentos completos: {features['patient'].nunique():,}")
print(f"Segmentos analisados: {len(features):,}")
print(f"Derivações encontradas: {sorted(features['n_leads'].unique())}")
print(f"Frequências de amostragem (Hz): {sorted(features['fs_hz'].unique())}")
display(features[["n_samples", "duration_hours", "mean", "std", "min", "max", "peak_to_peak", "n_annotations"]].describe().T.round(3))

segments_per_patient = features.groupby("patient").size().rename("segments")
duration_per_patient = features.groupby("patient")["duration_hours"].sum().rename("total_hours")
patient_summary = pd.concat([segments_per_patient, duration_per_patient], axis=1)
display(patient_summary.describe().round(2))
display(pd.Series(beat_counts, name="annotation_count").sort_values(ascending=False).to_frame().head(20))
rhythm_summary = pd.DataFrame({
    "regions": pd.Series(rhythm_region_counts),
    "duration_hours": pd.Series(rhythm_duration_samples) / (features["fs_hz"].iloc[0] * 3600),
}).fillna(0).sort_values("duration_hours", ascending=False)
display(rhythm_summary)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
sns.histplot(segments_per_patient, bins=30, ax=axes[0, 0])
axes[0, 0].set_title("Número de segmentos por paciente")
axes[0, 0].set_xlabel("Segmentos")
sns.histplot(patient_summary["total_hours"], bins=30, ax=axes[0, 1])
axes[0, 1].set_title("Duração total observada por paciente")
axes[0, 1].set_xlabel("Horas")
sns.histplot(features["std"], bins=50, ax=axes[1, 0])
axes[1, 0].set_title("Dispersão da amplitude por segmento")
axes[1, 0].set_xlabel("Desvio-padrão (unidade WFDB)")
beat_descriptions = {
    "N": "Normal",
    "S": "Supraventricular prematuro",
    "V": "Ventricular prematuro",
    "Q": "Não classificável",
}
top_beats = pd.Series({symbol: beat_counts.get(symbol, 0) for symbol in beat_descriptions})
beat_colors = {"N": "#4C78A8", "Q": "#F58518", "S": "#54A24B", "V": "#E45756"}
beat_percentages = top_beats / top_beats.sum() * 100
pie_labels = [f"{symbol} — {beat_descriptions[symbol]} (n={count:,})" for symbol, count in top_beats.items()]
pie_colors = [beat_colors.get(symbol, "#B279A2") for symbol in top_beats.index]
wedges, _ = axes[1, 1].pie(
    top_beats.values,
    colors=pie_colors,
    startangle=90,
    counterclock=False,
    radius=0.72,
    wedgeprops={"edgecolor": "white", "linewidth": 1},
)
small_wedges = []
for wedge, symbol, percentage in zip(wedges, top_beats.index, beat_percentages):
    angle = np.deg2rad((wedge.theta1 + wedge.theta2) / 2)
    x, y = np.cos(angle), np.sin(angle)
    label = f"{percentage:.1f}%"
    if symbol in {"N", "Q"}:
        axes[1, 1].text(0.48 * x, 0.48 * y, label, ha="center", va="center", color="white", fontsize=9, fontweight="bold")
    else:
        small_wedges.append((wedge, symbol, percentage, x, y))
small_wedges.sort(key=lambda item: item[1])
label_y_positions = np.linspace(0.72, -0.72, len(small_wedges)) if small_wedges else []
for index, ((wedge, symbol, percentage, x, y), label_y) in enumerate(zip(small_wedges, label_y_positions)):
    side = 1 if index % 2 == 0 else -1
    edge_x, edge_y = 0.72 * x, 0.72 * y
    outer_x, outer_y = 0.94 * x, 0.94 * y
    lane_x = 1.15 * side
    label_x = 1.22 * side
    line_color = beat_colors.get(symbol, "#555555")
    axes[1, 1].plot([edge_x, outer_x, lane_x, lane_x, label_x], [edge_y, outer_y, outer_y, label_y, label_y], color=line_color, linewidth=1.1, clip_on=False)
    axes[1, 1].text(label_x, label_y, f"{symbol}: {percentage:.1f}%", ha="left" if side > 0 else "right", va="center", fontsize=9, color=line_color, clip_on=False)
axes[1, 1].set_xlim(-1.65, 1.65)
axes[1, 1].set_ylim(-1.12, 1.12)
axes[1, 1].set_title("Proporção das anotações de batimento")
axes[1, 1].legend(pie_labels, title="Tipo de batimento e contagem", loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=1, fontsize=8, frameon=False)
plt.tight_layout(rect=[0, 0.08, 1, 1], w_pad=2.0, h_pad=2.0)
plt.show()

# Visualizar batimentos e intervalos de ritmo em segmentos representativos e extremos.
typical_row = features.iloc[(features["std"] - features["std"].median()).abs().argmin()]
extreme_row = features.loc[features["peak_to_peak"].idxmax()]
fig, axes = plt.subplots(2, 1, figsize=(15, 8))
for ax, row, title, center_on_peak in [
    (axes[0], typical_row, "Segmento próximo da dispersão mediana", False),
    (axes[1], extreme_row, "Segmento com maior amplitude pico-a-pico", True),
]:
    patient_dir = next(p for p in selected if p.endswith(row["patient"]))
    record_path = DOWNLOAD_DIR / patient_dir / row["record"]
    rec = wfdb.rdrecord(str(record_path))
    ann = wfdb.rdann(str(record_path), "atr")
    signal = np.asarray(rec.p_signal[:, 0], dtype=np.float64)
    fs = float(rec.fs)
    start = max(0, int(np.argmax(np.abs(signal))) - int(2 * fs)) if center_on_peak else 0
    stop = min(len(signal), start + int(10 * fs))
    ax.plot(np.arange(start, stop) / fs, signal[start:stop], linewidth=0.7, color="steelblue")
    active_rhythm, rhythm_start = None, None
    for sample, symbol, raw_note in zip(ann.sample, ann.symbol, ann.aux_note):
        sample, note = int(sample), str(raw_note).strip()
        if note.startswith("("):
            if active_rhythm is not None:
                left, right = max(start, rhythm_start), min(sample, stop)
                if left < right:
                    ax.axvspan(left / fs, right / fs, color="orange", alpha=0.16)
            active_rhythm, rhythm_start = note[1:].strip().rstrip(")").strip(), sample
        elif note == ")" and active_rhythm is not None:
            left, right = max(start, rhythm_start), min(sample, stop)
            if left < right:
                ax.axvspan(left / fs, right / fs, color="orange", alpha=0.16)
            active_rhythm, rhythm_start = None, None
        if start <= sample < stop and str(symbol).strip() not in {"+", "~"}:
            ax.axvline(sample / fs, color="crimson", alpha=0.22, linewidth=0.7)
    if active_rhythm is not None:
        left, right = max(start, rhythm_start), stop
        if left < right:
            ax.axvspan(left / fs, right / fs, color="orange", alpha=0.16)
    ax.set_title(f"{title}: {row['patient']} / {row['record']}")
    ax.set_ylabel("Amplitude (mV)")
    ax.set_xlabel("Tempo (s)")
    del rec, ann, signal
plt.tight_layout()
plt.show()
display(features.nlargest(10, "peak_to_peak")[
    ["patient", "record", "mean", "std", "min", "max", "peak_to_peak"]
].round(3))

print("Interpreta os resultados como EDA da amostra aleatória de pacientes, não dos 11.000 pacientes completos.")

## Resumo e interpretação do EDA

| Aspeto | Resultado observado |
|---|---:|
| Pacientes analisados | 20 |
| Segmentos completos analisados | 981 |
| Derivações e frequência | 1 derivação; 250 Hz |
| Duração por segmento | 1.048.577 amostras (cerca de 1,165 h) |
| Duração total observada | cerca de 1.142,95 h |
| Segmentos por paciente | média 49,05; mediana 50; mínimo 40; máximo 50 |
| Desvio-padrão da amplitude por segmento | mediana 0,125; percentil 95 1,972; máximo 25,027 |
| Segmentos com amplitude pico-a-pico > 100 mV | 67 de 981 (6,8%) |
| Anotações de batimento | `N` normal: 3.768.081; `Q` não classificável: 1.196.213; `S` supraventricular prematuro: 47.889; `V` ventricular prematuro: 45.948 |
| Regiões de ritmo identificadas | `N`: 30.605 regiões (840,47 h); `AFIB`: 2.012 regiões (43,55 h) |

Os 20 pacientes têm quantidades de segmentos relativamente próximas: 19 têm entre 40 e 50 segmentos e a média é 49,05. Todos os segmentos analisados têm a mesma frequência, número de amostras e duração, pelo que estas dimensões não variam nesta amostra. O gráfico circular mostra apenas anotações de batimento: `N` corresponde a batimento normal, `S` a batimento supraventricular prematuro, `V` a batimento ventricular prematuro e `Q` a batimento não classificável. O símbolo `+` não é um batimento e não faz parte do gráfico nem do cálculo das percentagens. Para os ritmos, foram consideradas apenas as notas de início (`(N`, `(AFIB`) e o fecho (`)`), agregando marcadores equivalentes como `(N)`; campos vazios/`None` não são tratados como categorias. As durações são somas dentro dos segmentos e não prevalências clínicas por paciente.

A distribuição da amplitude é assimétrica e contém valores extremos: 67 segmentos têm amplitude pico-a-pico acima de 100 mV, enquanto a mediana é cerca de 3,71 mV; o máximo chega a 881,21 mV. Os gráficos sobrepõem os eventos e as regiões de ritmo anotadas a um segmento típico e ao segmento com maior amplitude. Estes valores justificam verificar a escala/unidade e inspecionar os segmentos extremos antes de modelar; o EDA, por si só, não determina se resultam de artefactos, saturação ou sinal clinicamente relevante, pelo que não foram removidos.

Estes resultados descrevem apenas a amostra aleatória de 20 pacientes (981 dos 541.794 segmentos aproximados do conjunto completo), não uma estimativa exaustiva do Icentia11k. A amostra é adequada para exploração inicial do formato e da variabilidade, mas é pequena para conclusões representativas sobre todos os pacientes.